# Academic-Data-Agent

## Описание проекта

Это упрощённый демонстрационный Notebook для сообщества Hello-Agents, который быстро показывает, как Academic-Data-Agent обрабатывает два типа входных данных:

- структурированные табличные данные
- текстовые PDF-документы

Notebook по умолчанию работает в лёгком режиме для быстрого воспроизведения при ревью сообщества.

## Информация об авторе

- GitHub: @healer-666


## Настройка окружения

Перед первым запуском выполните в каталоге проекта:

```bash
pip install -r requirements.txt
```

и создайте `.env` на основе `.env.example`, указав рабочую конфигурацию модели.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
OUTPUT_ROOT.mkdir(exist_ok=True)

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from data_analysis_agent.agent_runner import run_analysis
from data_analysis_agent.presentation import render_trace_table, render_full_report, render_diagnostics


In [2]:
def render_run_summary(result, title: str):
    methods = ", ".join(result.methods_used) if result.methods_used else "unknown"
    tools = ", ".join(result.tools_used) if result.tools_used else "unknown"
    summary_md = f"""
## {title}

- Тип входных данных: `{result.input_kind}`
- Путь к данным: `{result.data_context.absolute_path.as_posix()}`
- Размер данных: `{result.data_context.shape[0]} x {result.data_context.shape[1]}`
- Определённая область: `{result.detected_domain}`
- Использованные инструменты: `{tools}`
- Методы анализа: `{methods}`
- Статус разбора документа: `{result.document_ingestion_status}`
- Количество кандидатных таблиц: `{result.candidate_table_count}`
- Выбранная основная таблица: `{result.selected_table_id or 'not_applicable'}`
- Мультитабличный режим PDF: `{result.pdf_multi_table_mode}`
- Путь к отчёту: `{result.report_path.as_posix()}`
- Путь к trace: `{result.trace_path.as_posix()}`
- Статус ревью: `{result.review_status}`
- Общее время: `{result.total_duration_ms / 1000:.2f}s`
"""
    display(Markdown(summary_md))


## Пример 1: анализ табличных данных

Здесь используется лёгкий Excel-образец для демонстрации стандартного пути анализа таблиц. Версия для сообщества по умолчанию использует `draft + auto`, чтобы минимизировать время ожидания.


In [3]:
tabular_result = run_analysis(
    data_path=PROJECT_ROOT / "data" / "sample_table.xlsx",
    output_dir=OUTPUT_ROOT,
    quality_mode="draft",
    latency_mode="auto",
    verbose=True,
)
render_run_summary(tabular_result, "Краткая сводка запуска табличного примера")


[output cleared — rerun cell after translation]


[output cleared — rerun cell after translation]


[output cleared — rerun cell]


In [4]:
render_trace_table(tabular_result)


[output cleared — rerun cell]


## Пример 2: анализ PDF-литературы

Здесь используется небольшой образец статьи. Текущая версия:

- извлекает контекст публикации
- определяет кандидатные таблицы
- автоматически выбирает одну основную таблицу для количественного анализа
- использует остальные кандидатные таблицы как контекстные доказательства при интерпретации отчёта


In [5]:
pdf_result = run_analysis(
    data_path=PROJECT_ROOT / "data" / "sample_paper.pdf",
    output_dir=OUTPUT_ROOT,
    quality_mode="draft",
    latency_mode="auto",
    document_ingestion_mode="auto",
    verbose=True,
)
render_run_summary(pdf_result, "Краткая сводка запуска PDF-примера")


[output cleared — rerun cell after translation]


[output cleared — rerun cell]


In [6]:
render_full_report(pdf_result)


[output cleared — rerun cell]


In [7]:
render_diagnostics(pdf_result)


[output cleared — rerun cell]


## Итоги и перспективы

Эта демонстрация для сообщества показывает три ключевые вещи:

- автоматический анализ структурированных таблиц
- извлечение кандидатных таблиц из PDF и анализ основной таблицы
- отслеживаемость артефактов запуска, отчётов и trace

Полная версия проекта также предоставляет Gradio-рабочее место, просмотр истории, визуальное ревью и более полные инженерные возможности.
